In [1]:
import sys
import os
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.hospital_env import HospitalEnv

data = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "synthetic_hospital_data.csv"))


STATE_DIM = 5
ACTIONS = [0, 1]

def normalize_state(state):
    arrival, slot, priority, no_show, icu_ratio = state
    return np.array([
        arrival / 480.0,
        slot / 480.0,
        priority / 2.0,
        no_show,
        icu_ratio
    ], dtype=np.float32)

# Linear weights
weights = {
    0: np.zeros(STATE_DIM),
    1: np.zeros(STATE_DIM)
}

def Q_value(state, action):
    return np.dot(weights[action], state)


# Hyperparameters

alpha = 0.005
gamma = 0.9
epsilon_start = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 100

epsilon = epsilon_start

episode_rewards = []
episode_waiting = []
episode_util = []


# Training Loop

for ep in range(episodes):

    env = HospitalEnv(data)
    state = normalize_state(env.reset())
    done = False

    total_reward = 0
    total_wait = 0
    total_util = 0
    steps = 0

    while not done:

        if np.random.rand() < epsilon:
            action = np.random.choice(ACTIONS)
        else:
            q_vals = [Q_value(state, a) for a in ACTIONS]
            action = ACTIONS[np.argmax(q_vals)]

        next_state, reward, done, info = env.step(action)

        total_reward += reward
        total_wait += info["waiting_time"]
        total_util += info["resource_util"]
        steps += 1

        if not done:
            next_state_norm = normalize_state(next_state)
            target = reward + gamma * max(
                Q_value(next_state_norm, a) for a in ACTIONS
            )
        else:
            target = reward

        prediction = Q_value(state, action)
        error = target - prediction

        weights[action] += alpha * error * state

        if not done:
            state = next_state_norm

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    episode_rewards.append(total_reward)
    episode_waiting.append(total_wait / steps)
    episode_util.append(total_util / steps)

    print(f"Episode {ep+1}: Reward = {total_reward}")

print("\nTraining completed.")


# Save Logs

results = pd.DataFrame({
    "episode": range(1, episodes + 1),
    "reward": episode_rewards,
    "waiting_time": episode_waiting,
    "resource_util": episode_util
})

results.to_csv("linear_q_training_log.csv", index=False)


# Plot Reward Curve

plt.figure()
plt.plot(episode_rewards)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Linear Q-Learning Training Curve")
plt.savefig("linear_q_reward_curve.png")
plt.close()

Episode 1: Reward = -2005
Episode 2: Reward = -2065
Episode 3: Reward = -1387
Episode 4: Reward = -2030
Episode 5: Reward = -1461
Episode 6: Reward = -2285
Episode 7: Reward = -1973
Episode 8: Reward = -2002
Episode 9: Reward = -1626
Episode 10: Reward = -1376
Episode 11: Reward = -1656
Episode 12: Reward = -2064
Episode 13: Reward = -1457
Episode 14: Reward = -1518
Episode 15: Reward = -2131
Episode 16: Reward = -1325
Episode 17: Reward = -1762
Episode 18: Reward = -2410
Episode 19: Reward = -1496
Episode 20: Reward = -1179
Episode 21: Reward = -1354
Episode 22: Reward = -1853
Episode 23: Reward = -1897
Episode 24: Reward = -1580
Episode 25: Reward = -2313
Episode 26: Reward = -1559
Episode 27: Reward = -1327
Episode 28: Reward = -1496
Episode 29: Reward = -1645
Episode 30: Reward = -1583
Episode 31: Reward = -1737
Episode 32: Reward = -1317
Episode 33: Reward = -1382
Episode 34: Reward = -1281
Episode 35: Reward = -2114
Episode 36: Reward = -1755
Episode 37: Reward = -1095
Episode 38

In [2]:
import pandas as pd


# Load training log

df = pd.read_csv("linear_q_training_log.csv")


# Load convergence

conv = pd.read_csv("linear_q_convergence.csv")


avg_reward = df["reward"].tail(20).mean()
avg_wait = df["waiting_time"].tail(20).mean()
avg_util = df["resource_util"].tail(20).mean()
conv_ep = conv["convergence_episode"][0]


performance_row = pd.DataFrame({
    "Algorithm": ["Linear Q-Learning"],
    "Avg Reward": [avg_reward],
    "Waiting Time": [avg_wait],
    "Resource Utilization": [avg_util],
    "Convergence Episode": [conv_ep]
})

performance_row.to_csv("linear_q_performance_row.csv", index=False)

print("Linear Q performance row created.")
print(performance_row)

Linear Q performance row created.
           Algorithm  Avg Reward  Waiting Time  Resource Utilization  \
0  Linear Q-Learning    -1191.95        54.534                84.642   

   Convergence Episode  
0                   76  
